# Streaming Text and Events with the Anthropic API

This notebook demonstrates how to use Claude's streaming API for real-time output. You'll learn:

- Why streaming improves user experience
- Basic text streaming with `text_stream`
- Processing raw server-sent events
- Extracting token usage from streams
- Buffering streamed output to a string
- Using stop sequences with streaming
- Async streaming with `AsyncAnthropic`
- Progress indicator patterns
- Handling stream interruptions gracefully

## 1. Setup

Install the Anthropic SDK and import it. The client reads your `ANTHROPIC_API_KEY` environment variable automatically — no hardcoded keys needed.

In [ ]:
%pip install anthropic --quiet

In [ ]:
import anthropic
import asyncio
import time
import sys

# Sync client — reads ANTHROPIC_API_KEY from the environment
client = anthropic.Anthropic()

# Async client — used later for async streaming examples
async_client = anthropic.AsyncAnthropic()

MODEL = "claude-haiku-4-5"
print(f"Using model: {MODEL}")

## 2. Why Streaming?

Without streaming, your code must wait for the **entire response** before showing anything to the user. For long answers this means seconds of silence.

With streaming, the first tokens appear almost immediately — the model starts outputting while it is still generating the rest. This dramatically reduces perceived latency.

```
Non-streaming timeline:
  [request] ──── 2.4 s of silence ────> [entire response displayed at once]

Streaming timeline:
  [request] ─> [token 1] ─> [token 2] ─> [token 3] ─> ... ─> [last token]
                 ~0.3 s        ~0.35 s      ~0.4 s
```

The total generation time is the same, but users see progress instead of a blank screen.

## 3. Basic Text Streaming

The simplest streaming pattern: iterate over `stream.text_stream` and print each chunk as it arrives.

`end=""` and `flush=True` keep the output on one continuous line without buffering.

In [ ]:
print("Streaming response:\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Explain what a transformer neural network is in 3 sentences."}
    ],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n[Stream complete]")

**Expected streamed output (mock):**

```
Streaming response:

A transformer neural network is a deep learning architecture that uses self-attention
 mechanisms to weigh the importance of different parts of the input sequence when
 producing each output token. Unlike recurrent networks, transformers process the
 entire sequence in parallel, making them highly efficient on modern GPUs. They
 underpin most state-of-the-art language models, including GPT and Claude.

[Stream complete]
```

Each fragment above (`A transformer`, ` neural network`, ` is a deep...`) arrives as a separate chunk from the API.

## 4. Accessing Raw Events

Iterating over `stream` directly gives you every server-sent event. This is useful when you need full control — for example, updating a UI widget on `content_block_delta` or logging billing metadata from `message_delta`.

The event types you'll see:

| Event type | When it fires |
|---|---|
| `message_start` | Once at the beginning, carries the message ID and initial usage |
| `content_block_start` | When a new content block opens (e.g., a text block) |
| `content_block_delta` | Each incremental text chunk |
| `content_block_stop` | When the content block closes |
| `message_delta` | Final token counts and stop reason |
| `message_stop` | End-of-stream sentinel |

In [ ]:
print("Raw event log:\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=64,
    messages=[
        {"role": "user", "content": "Say 'hello world' and nothing else."}
    ],
) as stream:
    for event in stream:
        print(f"  type={event.type!r}", end="")
        # Print delta text when available
        if hasattr(event, "delta") and hasattr(event.delta, "text"):
            print(f"  text={event.delta.text!r}", end="")
        # Print usage when available
        if hasattr(event, "usage"):
            print(f"  usage={event.usage}", end="")
        print()

**Expected event log (mock):**

```
Raw event log:

  type='message_start'         usage=Usage(input_tokens=14, output_tokens=1)
  type='content_block_start'
  type='content_block_delta'   text='hello'
  type='content_block_delta'   text=' world'
  type='content_block_stop'
  type='message_delta'         usage=Usage(output_tokens=3)
  type='message_stop'
```

Notice that `content_block_delta` events carry the incremental `text`, while `message_delta` carries the final token count.

## 5. Extracting Token Usage

Call `stream.get_final_message()` after the context manager exits to retrieve the fully assembled `Message` object, including input and output token counts. This is the recommended way to get accurate billing data from a stream.

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=128,
    messages=[
        {"role": "user", "content": "List three benefits of using async I/O in Python."}
    ],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

# Fetch the final assembled message once the stream is done
final_message = stream.get_final_message()
usage = final_message.usage

print(f"\n\n--- Token Usage ---")
print(f"Input tokens:  {usage.input_tokens}")
print(f"Output tokens: {usage.output_tokens}")
print(f"Total tokens:  {usage.input_tokens + usage.output_tokens}")

**Expected output (mock):**

```
1. Higher throughput — async I/O lets a single thread handle many connections simultaneously.
2. Lower memory usage — no need to spawn a thread per connection.
3. Cleaner concurrency — coroutines are easier to reason about than threads.

--- Token Usage ---
Input tokens:  18
Output tokens: 47
Total tokens:  65
```

## 6. Streaming to a String Buffer

Sometimes you need the complete text without printing mid-stream — for example, to post-process it or store it in a variable. Use `stream.get_final_text()`, which blocks until the stream finishes and returns the assembled string.

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=128,
    messages=[
        {"role": "user", "content": "Give me a one-sentence definition of entropy."}
    ],
) as stream:
    # Silently consume the stream, then return the complete text
    full_text = stream.get_final_text()

# Work with the full text after streaming is complete
print("Full response captured:")
print(full_text)
print(f"\nCharacter count: {len(full_text)}")
print(f"Word count:      {len(full_text.split())}")

**Expected output (mock):**

```
Full response captured:
Entropy is a thermodynamic quantity that measures the degree of disorder or randomness
in a system, with higher entropy indicating greater disorder.

Character count: 141
Word count:      24
```

Use `get_final_text()` when downstream logic (regex, JSON parsing, logging) should only run on the complete string.

## 7. Streaming with Stop Sequences

Pass `stop_sequences` to terminate the stream early when a sentinel token appears. The stream stops as soon as the model outputs the stop sequence — tokens after it are never generated, saving cost and latency.

In [ ]:
print("Streaming until STOP sentinel:\n")

with client.messages.stream(
    model=MODEL,
    max_tokens=256,
    stop_sequences=["STOP"],
    messages=[
        {
            "role": "user",
            "content": (
                "Count slowly from 1 upward. After each number write a newline. "
                "After the number 5, write the word STOP and then keep going."
            ),
        }
    ],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

final = stream.get_final_message()
print(f"\n\nStop reason: {final.stop_reason}")
print(f"Stop sequence triggered: {final.stop_sequence!r}")

**Expected output (mock):**

```
Streaming until STOP sentinel:

1
2
3
4
5

Stop reason: stop_sequence
Stop sequence triggered: 'STOP'
```

The stream terminated at `STOP`. Tokens for `6`, `7`, ... were never sent over the wire.

## 8. Async Streaming

For production applications (web servers, async pipelines), use `AsyncAnthropic` so the stream does not block the event loop. The API is identical to the sync version but prefixed with `async`/`await`.

In [ ]:
async def stream_async_response(prompt: str) -> str:
    """Stream a response asynchronously and return the full text."""
    collected_chunks = []

    async with async_client.messages.stream(
        model=MODEL,
        max_tokens=128,
        messages=[{"role": "user", "content": prompt}],
    ) as stream:
        async for text in stream.text_stream:
            print(text, end="", flush=True)
            collected_chunks.append(text)

    return "".join(collected_chunks)


print("Async streaming response:\n")
result = await stream_async_response(
    "Name three programming languages invented in the 1990s, one per line."
)
print(f"\n\nAsync stream collected {len(result)} characters.")

**Expected output (mock):**

```
Async streaming response:

Python
Ruby
Java

Async stream collected 18 characters.
```

> **Note:** In a Jupyter notebook, `await` works at the top level. In a regular Python script, wrap the call in `asyncio.run(stream_async_response(prompt))`.

## 9. Progress Indicator Pattern

Because you receive tokens one-by-one, you can track progress in real time. Here we count characters and render a simple inline progress bar that updates as the stream arrives.

In [ ]:
def render_progress_bar(chars_received: int, expected_chars: int = 400, width: int = 30) -> str:
    """Return a simple ASCII progress bar string."""
    fraction = min(chars_received / expected_chars, 1.0)
    filled = int(fraction * width)
    bar = "#" * filled + "-" * (width - filled)
    pct = int(fraction * 100)
    return f"\r[{bar}] {pct:3d}%  ({chars_received} chars)"


print("Streaming with live progress bar:\n")
chars = 0
full_chunks = []

with client.messages.stream(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "Write a short paragraph about the history of the internet."}
    ],
) as stream:
    for text in stream.text_stream:
        full_chunks.append(text)
        chars += len(text)
        sys.stdout.write(render_progress_bar(chars))
        sys.stdout.flush()

# Move to a new line after the progress bar
print()
print("\nFull response:")
print("".join(full_chunks))

**Expected output (mock — progress bar frames):**

```
Streaming with live progress bar:

[##----------------------------]  7%  (28 chars)
[######------------------------] 20%  (81 chars)
[##############----------------] 47%  (188 chars)
[##############################] 100%  (400 chars)

Full response:
The internet traces its roots to ARPANET, a 1969 US Defense Department project
that first connected computers across universities. Tim Berners-Lee's 1991
invention of the World Wide Web transformed it from an academic tool into a
global communication platform, and broadband adoption in the 2000s made
always-on connectivity the norm worldwide.
```

The `\r` carriage return overwrites the same terminal line on each update, creating an animated bar effect.

## 10. Handling Interruptions

Network errors or early exits can interrupt a stream. Wrap the streaming loop in `try/except` to catch `anthropic.APIConnectionError`, `anthropic.APIStatusError`, and `KeyboardInterrupt`. After an interruption, check `stream.response.is_closed` to see whether the connection was already closed.

In [ ]:
def stream_with_interruption_handling(prompt: str, max_tokens: int = 256) -> str:
    """Stream a response with graceful error handling."""
    collected = []

    try:
        with client.messages.stream(
            model=MODEL,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        ) as stream:
            for text in stream.text_stream:
                print(text, end="", flush=True)
                collected.append(text)

    except KeyboardInterrupt:
        print("\n[Stream interrupted by user]")
        is_closed = getattr(getattr(stream, "response", None), "is_closed", True)
        print(f"Connection closed: {is_closed}")

    except anthropic.APIConnectionError as e:
        print(f"\n[Connection error: {e}]")

    except anthropic.APIStatusError as e:
        print(f"\n[API error {e.status_code}: {e.message}]")

    finally:
        partial = "".join(collected)
        if partial:
            print(f"\n[Partial text captured: {len(partial)} characters]")

    return "".join(collected)


print("Streaming with error handling:\n")
result = stream_with_interruption_handling(
    "Describe the water cycle in two sentences."
)
print(f"\nFinal captured length: {len(result)} chars")

**Expected output (mock — successful run):**

```
Streaming with error handling:

Water evaporates from oceans and lakes, rises as vapour, condenses into clouds,
and falls back as precipitation. It then flows via rivers and groundwater back
to the oceans, completing the cycle.

[Partial text captured: 174 characters]

Final captured length: 174 chars
```

If you press **Stop** mid-cell, `KeyboardInterrupt` is caught and whatever text arrived before the interrupt is returned.

## 11. Summary

### When to use `text_stream` vs. raw event iteration

| Use case | Recommended approach |
|---|---|
| Print text to terminal as it arrives | `for text in stream.text_stream` |
| Capture full response silently | `stream.get_final_text()` |
| React to specific event types (e.g., billing, UI updates) | `for event in stream` |
| Get token counts | `stream.get_final_message().usage` |
| Stop early on a sentinel token | `stop_sequences=[...]` |
| Non-blocking async pipeline | `async with async_client.messages.stream(...)` |

### Sync vs. Async

- **Sync** (`anthropic.Anthropic`) — fine for scripts, notebooks, and CLI tools where blocking is acceptable.
- **Async** (`anthropic.AsyncAnthropic`) — required for async web frameworks (FastAPI, aiohttp) or any code that must handle concurrent requests without thread-per-connection overhead.

### Key takeaways

1. Streaming requires only one extra line compared to non-streaming: wrap `client.messages.create` with `client.messages.stream` in a `with` block.
2. The `stream.get_final_message()` call always works — even after iterating the stream — because the SDK buffers the assembled message.
3. Always handle `APIConnectionError` and `KeyboardInterrupt` in production to avoid silent data loss.
4. Stop sequences work the same way with streaming — the connection closes the moment the sentinel appears.